In [1]:
# Cell 1: Imports and config
import pandas as pd
import hashlib
from pathlib import Path

# ⚠️ Change this to something secret and memorable. Never share it.
SALT = "bbl_my_secret_salt_2024"

RAW_DIR = Path("raw_data")
OUT_DIR = Path("clean_data")
OUT_DIR.mkdir(exist_ok=True)

Intel MKL WARNING: Support of Intel(R) Streaming SIMD Extensions 4.2 (Intel(R) SSE4.2) enabled only processors has been deprecated. Intel oneAPI Math Kernel Library 2025.0 will require Intel(R) Advanced Vector Extensions (Intel(R) AVX) instructions.
Intel MKL WARNING: Support of Intel(R) Streaming SIMD Extensions 4.2 (Intel(R) SSE4.2) enabled only processors has been deprecated. Intel oneAPI Math Kernel Library 2025.0 will require Intel(R) Advanced Vector Extensions (Intel(R) AVX) instructions.


In [2]:
# Cell 2: DISCOVERY — run this first before filling in the column map
# Prints every column name from every xlsx file so you know what you're working with.
for xlsx_file in sorted(RAW_DIR.glob("*.xlsx")):
    df = pd.read_excel(xlsx_file, nrows=0)  # headers only, no data loaded
    print(f"\n📄 {xlsx_file.name}")
    for col in df.columns:
        print(f"   | {repr(col)}")


📄 202501_cell-gene-therapy.xlsx
   | 'Unnamed: 0'
   | 'Name \n(linkedin Profile)'
   | 'gmail'
   | 'Summary Linkedin Profile (Core Competencies)'
   | 'Companies (key role)'
   | 'Why interested in BBL?'
   | 'Question on Cell & Gene'
   | 'Would you like to present in future'
   | 'What are interested in learning about'
   | 'Job search\n(area, function, seniority)'
   | 'Known Job offers\n(area, function, seniority)'

📄 202502_ai-in-biotech.xlsx
   | '#'
   | 'Name'
   | 'Unnamed: 2'
   | 'Unnamed: 3'
   | 'email'
   | 'Summary Linkedin Profile (Core Competencies)'
   | 'Companies (key role)'
   | 'Why interested in BBL?'
   | '"Other" reason you are intersted in BBL?'
   | 'Do you have any specific questions(s) or feedback regarding the topic being discussed at this meeting?:'
   | 'Would you like to present in future'
   | 'Please add any future topics you would like to hear about?:'
   | 'Job search\n(area, function, seniority)'
   | 'Known Job offers\n(area, function, seniorit

In [3]:
# Cell 3: Hash function
def hash_email(email: str, salt: str) -> str:
    """Returns a consistent, anonymous ID for a given email."""
    if pd.isna(email) or str(email).strip() == "":
        return "unknown"
    combined = (salt + str(email).strip().lower()).encode()
    return hashlib.sha256(combined).hexdigest()[:12]  # 12-char ID is plenty

In [4]:
# Cell 4: Column name standardization map
# Built from the actual BBL files discovered in Cell 2.
# Columns NOT listed here pass through as-is (or get dropped in Cell 5 if identifying).

COLUMN_MAP = {
    # --- Email ---
    "email": "email",
    "gmail": "email",           # 202501 used 'gmail' as the column header

    # --- Name variants (all dropped after hashing) ---
    "Name": "name",
    "Name    ": "name",         # 202602 has trailing spaces
    'Name \n(LinkedIn link)    ': "name",   # 202603, 202604
    'Name \n(linkedin Profile)': "name",    # 202501 (lowercase)

    # --- Location ---
    "Location": "location",
    "Institution": "location",
    "Affiliation": "location",

    # --- Topic interest ---
    'What are interested in learning about': "topic_interest",
    'Please add any future topics you would like to hear about?:': "topic_interest",

    # --- Why BBL / community interest ---
    'Why interested in BBL?': "community_interest",
    '"Other" reason you are intersted in BBL?': "community_interest",

    # --- Wants to present ---
    'Would you like to present in future': "wants_to_present",
    'Is there a topic you would like to present at future events?:': "wants_to_present",

    # --- Job searching (useful aggregate signal, no personal info) ---
    'Currently job searching?:': "job_searching",
    'Job search\n(area, function, seniority)': "job_searching",

    # --- Company hiring ---
    'Is your company hiring?:': "company_hiring",
    'Known Job offers\n(area, function, seniority)': "company_hiring",

    # --- LinkedIn share consent ---
    "Authorize BBL": "authorize_share",

    # --- Mentoring interest ---
    'Interested in being a mentor?:': "mentoring_interest",
    'Are you interested in mentoring, and if yes, in which specific area?:': "mentoring_interest",
}

# Columns to always drop (identifying info)
DROP_ALWAYS = [
    "email", "name",
    "Summary Linkedin Profile (Core Competencies)",
    "Linkedin Profile Summary\n(Core Competencies)",
    "Additional Linkedin info",
    "Companies (key role)",
    "Companies (key roles)",
    "Confirmed in Master Directory",
    "Share email?",
    "Preference contact Linkedin/email (N= no preference)",
    'If you are currently looking for job opportunities, please provide details (company area/market, seniority, type etc):\t\n',
    'If your company currently has available job opportunities, please provide details (company area/market, seniority, type etc):\t',
    'If you are looking for SME, Consulting or other types of support, please provide details:',
]

In [9]:
# Cell 5: Process each xlsx file
all_events = []

for xlsx_file in sorted(RAW_DIR.glob("*.xlsx")):
    event_name = xlsx_file.stem  # rename your files descriptively before running!
    print(f"Processing: {event_name}")

    df = pd.read_excel(xlsx_file)

    # Rename columns to standard names
    df = df.rename(columns={k: v for k, v in COLUMN_MAP.items() if k in df.columns})

    # Remove duplicate column names (keep first — can happen when two source columns map to the same standard name)
    df = df.loc[:, ~df.columns.duplicated()]

    # Hash email → attendee_id
    if "email" in df.columns:
        df["attendee_id"] = df["email"].apply(lambda e: hash_email(e, SALT))
    else:
        print(f"  ⚠️ No email column found in {xlsx_file.name} — attendee_id will be 'unknown'")
        df["attendee_id"] = "unknown"

    # Keep only the standard columns we care about (drops all unmapped/junk columns)
    KEEP_COLUMNS = [
        "attendee_id", "event_name", "registered",
        "location", "topic_interest", "community_interest",
        "wants_to_present", "job_searching", "company_hiring",
        "mentoring_interest", "authorize_share",
    ]

    # Add event label before filtering so it's available
    df["event_name"] = event_name
    df["registered"] = True  # registration = attendance for BBL

    df = df[[c for c in KEEP_COLUMNS if c in df.columns]]

    all_events.append(df)
    print(f"  → {len(df)} rows | columns: {list(df.columns)}")

master = pd.concat(all_events, ignore_index=True)
print(f"\nTotal rows: {len(master)}")
print(master.head())

Processing: 202501_cell-gene-therapy
  → 47 rows | columns: ['attendee_id', 'event_name', 'registered', 'topic_interest', 'community_interest', 'wants_to_present', 'job_searching', 'company_hiring']
Processing: 202502_ai-in-biotech
  → 48 rows | columns: ['attendee_id', 'event_name', 'registered', 'location', 'topic_interest', 'community_interest', 'wants_to_present', 'job_searching', 'company_hiring']
Processing: 202504_multiomics
  → 52 rows | columns: ['attendee_id', 'event_name', 'registered', 'location', 'topic_interest', 'wants_to_present', 'mentoring_interest']
Processing: 202506_startup-conversations
  → 64 rows | columns: ['attendee_id', 'event_name', 'registered', 'location', 'job_searching', 'company_hiring', 'mentoring_interest']
Processing: 202508_protein-seq-and-app
  → 44 rows | columns: ['attendee_id', 'event_name', 'registered', 'location', 'job_searching', 'company_hiring', 'mentoring_interest']
Processing: 202510_tcell-therapy-and-bispecifics
  → 41 rows | columns: [

In [10]:
# Cell 6: Data check — review before saving
print("=== Columns in master ===")
print(master.columns.tolist())

print("\n=== Null counts ===")
print(master.isnull().sum())

print("\n=== Events ===")
print(master["event_name"].value_counts())

print("\n=== Unique attendee IDs ===")
print(master["attendee_id"].nunique())

print("\n=== Sample rows (no identifying info) ===")
print(master[["attendee_id", "event_name"] + 
      [c for c in ["location", "topic_interest", "authorize_share"] if c in master.columns]].head(5))

=== Columns in master ===
['attendee_id', 'event_name', 'registered', 'topic_interest', 'community_interest', 'wants_to_present', 'job_searching', 'company_hiring', 'location', 'mentoring_interest', 'authorize_share']

=== Null counts ===
attendee_id             0
event_name              0
registered              0
topic_interest        366
community_interest    396
wants_to_present      365
job_searching         372
company_hiring        403
location              286
mentoring_interest    414
authorize_share       400
dtype: int64

=== Events ===
event_name
202604_ai-in-biotech                    96
202506_startup-conversations            64
202504_multiomics                       52
202603_professional-networking-event    52
202502_ai-in-biotech                    48
202501_cell-gene-therapy                47
202602_navigating-fda-strategy          45
202508_protein-seq-and-app              44
202510_tcell-therapy-and-bispecifics    41
Name: count, dtype: int64

=== Unique attendee I

In [11]:
# Cell 7: Save — only run after Cell 6 looks clean
master.to_csv(OUT_DIR / "bbl_attendance_clean.csv", index=False)
print("Saved to clean_data/bbl_attendance_clean.csv")
print("✅ Verify: no names or emails below")
print(master.columns.tolist())

Saved to clean_data/bbl_attendance_clean.csv
✅ Verify: no names or emails below
['attendee_id', 'event_name', 'registered', 'topic_interest', 'community_interest', 'wants_to_present', 'job_searching', 'company_hiring', 'location', 'mentoring_interest', 'authorize_share']
